# 13 — Advanced Generation

Two advanced patterns:

| Pattern | Prompt | Purpose |
|---|---|---|
| Structured output | `STRUCTURED_OUTPUT_PROMPT` | Fixed schema (Q / A / summary / rows) |
| Reasoning | `REASONING_TEMPLATE` | Step-by-step before final answer |

**Config source:** `configs/default.yaml` → `notebooks.sample_question`

In [ ]:
from rag_pipeline.utils import load_notebook_config, format_docs
from rag_pipeline.embeddings import build_embeddings
from rag_pipeline.vectorstores import load_vectorstore
from rag_pipeline.retrieval import build_retriever
from rag_pipeline.generation import build_llms, PROMPT_REGISTRY
from rag_pipeline.generation.prompts import (
    STRUCTURED_OUTPUT_PROMPT, REASONING_TEMPLATE,
)

cfg, REPO = load_notebook_config()
FAISS_DIR = REPO / cfg.paths["faiss_index"]
QUESTION  = cfg.notebooks["sample_question"]

In [ ]:
emb = build_embeddings(dict(cfg.embeddings))
store = load_vectorstore(emb, {"type": "faiss", "persist_dir": str(FAISS_DIR)})
retriever = build_retriever(store, {"search_type": "similarity", "k": 3})

llms = build_llms(dict(cfg.llm))
model_name = next(iter(llms))
llm = llms[model_name]
print("Model:", model_name)

**Structured output**

In [ ]:
ctx = format_docs(retriever.invoke(QUESTION), max_chars=300)
prompt = STRUCTURED_OUTPUT_PROMPT.format(context=ctx, question=QUESTION)

print("=" * 60, "\nSTRUCTURED OUTPUT\n", "=" * 60, sep="")
for chunk in llm.stream(prompt, max_new_tokens=250):
    print(str(chunk), end="", flush=True)
print()

**Reasoning (step-by-step)**

In [ ]:
ctx = "\n".join(
    f"[Row {d.metadata.get('row', i)}] {d.page_content[:300]}..."
    for i, d in enumerate(retriever.invoke(QUESTION), 1)
)
prompt = REASONING_TEMPLATE.format(context=ctx, question=QUESTION)

print("=" * 60, "\nREASONING\n", "=" * 60, sep="")
for chunk in llm.stream(prompt, max_new_tokens=300):
    print(str(chunk), end="", flush=True)
print()

**Prompting strategy comparison (dict-driven)**

In [ ]:
NAMES = [n for n in ["rag", "cot", "react", "tot", "few_shot", "reasoning"]
         if n in PROMPT_REGISTRY]

for name in NAMES:
    template = PROMPT_REGISTRY[name]
    ctx = format_docs(retriever.invoke(QUESTION), max_chars=250)
    prompt = template.format(context=ctx, question=QUESTION)
    out = "".join(str(c) for c in llm.stream(prompt, max_new_tokens=180))
    print(f"\n{'='*60}\n{name.upper()}\n{'='*60}")
    print(out[:600] + ("..." if len(out) > 600 else ""))